## Direct inference with transformers

samples: https://www.kaggle.com/datasets/abdulvahap/music-instrunment-sounds-for-classification

In [ ]:
!pip install transformers datasets soundfile librosa torch

In [ ]:
from transformers import pipeline

# 2. Initialize the pre-trained audio classification pipeline
classifier = pipeline("audio-classification", model="dima806/musical_instrument_detection")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/2.27k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
result = classifier("sample1.wav")

# 4. Print the top predicted instruments and confidence scores
print(result)

[{'score': 0.6989988088607788, 'label': 'Sound_Piano'}, {'score': 0.1647523045539856, 'label': 'Sound_Guitar'}, {'score': 0.1362488716840744, 'label': 'Sound_Drum'}]


## Speechbrain test
https://huggingface.co/speechbrain/cnn14-esc50

In [ ]:
!pip install speechbrain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 18.2 MB/s eta 0:00:00


In [ ]:
from speechbrain.inference.classifiers import AudioClassifier

model = AudioClassifier.from_hparams(source="speechbrain/cnn14-esc50", savedir='pretrained_models/cnn14-esc50')

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/pretrained_models/cnn14-esc50/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch embedding_model_esc50ft.ckpt: Using symlink found at '/content/pretrained_models/cnn14-esc50/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier_esc50.ckpt: Using symlink found at '/content/pretrained_models/cnn14-esc50/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/pretrained_models/cnn14-esc50/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, classifier, label_encoder


In [ ]:
out_probs, score, index, text_lab = model.classify_file('sample1_piano.wav')
print(text_lab)

Resampling the audio from 22050 Hz to 44100 Hz
[['door_wood_creaks']]


In [ ]:
out_probs, score, index, text_lab = model.classify_file('sample4_trumpet.wav')
print(text_lab)

Resampling the audio from 22050 Hz to 44100 Hz
[['insects']]


Result: very poor perfomance on musical instruments classification in speechbrain

## Use PANNs Inference
This is currently the best as per my experience

In [1]:
!pip install -q panns-inference librosa

In [2]:
import librosa
from panns_inference import AudioTagging

# 1. Load the general-purpose audio tagger
model = AudioTagging(checkpoint_path=None, device='cuda:0') # Downloads automatically

Checkpoint path: /root/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


In [7]:
# audio_path="sample1_piano.wav"
# audio_path="sample2_flute.wav"
# audio_path = "sample3_flute.wav"
audio_path = "sample4_trumpet.wav"

x, sr = librosa.load(audio_path, sr=32000, mono=True)
x = x[None, :] # Add batch dimension

# 3. Run inference
print_top_k = 5
predict_output = model.inference(x)

# 4. Parse results to find 'Harmonica'
for tuple_set in predict_output[0]:
    # Looking through the top classes sorted by probability
    top_indices = tuple_set.argsort()[-print_top_k:][::-1]
    for idx in top_indices:
        print(f"Class: {model.labels[idx]}, Probability: {tuple_set[idx]:.4f}")

Class: Music, Probability: 0.6459
Class: Trumpet, Probability: 0.6077
Class: Brass instrument, Probability: 0.5847
Class: Trombone, Probability: 0.5230
Class: Musical instrument, Probability: 0.2349
